# 02 — Training Benchmark: Parquet vs Lance

**Purpose:** Read each format back through Ray Data and feed it to Ray Train, measuring where the storage format separates the two pipelines. Covers stages **4 (inline preprocess), 5 (train), 6 (compare)** of [`README.md`](README.md).

**Prerequisite:** Run `01_create_benchmark_datasets.ipynb` for the target `size` tier first.

| Section | What it measures |
|---------|------------------|
| **1 — Data loading throughput** | Pure read + decode/resize samples/sec; full-column vs projected read |
| **2 — Training throughput** | Dummy-model (I/O-bound) and real ResNet-50 (compute-attributed) runs; samples/sec, time-to-first-batch, per-batch latency distribution |
| **3 — Compare** | Side-by-side, logged to MLflow |

**Decode/resize is fused into the read path** (not materialized) — once decoded to fixed-size tensors both formats store identical arrays, which erases the variable-blob difference that is the whole point of the comparison.

---

### GPU sizing

The model is a **transfer-learning ResNet-50** classifier — deliberately light compute. On an **A10 (24GB)** it runs ~600–900 img/s/GPU at 224px with AMP, so a single GPU is easy to *starve*. That is exactly what surfaces a data-loading bottleneck: if the storage format can't feed the GPU, utilization drops and it shows up in samples/sec.

Starting point: **4 × A10** (`num_workers=4`, one A10 each) → ~2,400–3,600 img/s aggregate GPU demand, the range where shuffled random-access read throughput starts to diverge between Lance and Parquet. Bump `num_gpu_workers` to 8 for the 1m/10m tiers.

In [ ]:
# Install BEFORE any ray init. Ray ships on DBR ML 15.0+; pin for reproducibility.
%pip install -qU "ray[data,train]==2.54.0" "lance==0.17.0" "pyarrow>=16.0" torch torchvision Pillow numpy pandas "mlflow<3.0,>=2.17"
dbutils.library.restartPython()

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("num_gpu_workers", "4", "GPU workers (A10)")
dbutils.widgets.text("batch_size", "64", "Batch size")
dbutils.widgets.text("num_epochs", "3", "Epochs (steady-state)")
dbutils.widgets.text("mlflow_experiment", "", "MLflow experiment (blank = default)")

size            = dbutils.widgets.get("size")
catalog         = dbutils.widgets.get("catalog")
schema          = dbutils.widgets.get("schema")
volume          = dbutils.widgets.get("volume")
NUM_GPU_WORKERS = int(dbutils.widgets.get("num_gpu_workers"))
BATCH_SIZE      = int(dbutils.widgets.get("batch_size"))
NUM_EPOCHS      = int(dbutils.widgets.get("num_epochs"))

# MUST match 01_create_benchmark_datasets.ipynb.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]
IMG_SIZE   = 224

base_vol     = f"/Volumes/{catalog}/{schema}/{volume}"
parquet_path = f"{base_vol}/synthetic_parquet_{size}"
lance_path   = f"{base_vol}/synthetic_lance_{size}"
ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
username      = notebook_path.split("/")[2]
mlflow_exp    = dbutils.widgets.get("mlflow_experiment") or f"/Users/{username}/parquet-vs-lance-benchmark"

print(f"Size tier   : {size}")
print(f"GPU workers : {NUM_GPU_WORKERS} x A10")
print(f"Parquet     : {parquet_path}")
print(f"Lance       : {lance_path}")
print(f"MLflow exp  : {mlflow_exp}")

In [ ]:
import os

# Credentials — set BEFORE cluster setup so Ray workers inherit them (Ray 2.41+).
os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [ ]:
# GPU Ray cluster — PATH A. Set spark.task.resource.gpu.amount = "0" in the cluster
# Spark config so Ray (not Spark) controls GPU allocation. One A10 per worker node.
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(
    min_worker_nodes=NUM_GPU_WORKERS,
    max_worker_nodes=NUM_GPU_WORKERS,      # fixed size
    num_gpus_worker_node=1,                # one A10 per node
    num_cpus_worker_node=16,               # match node CPU count (feeds the decode pipeline)
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

total_gpus = ray.cluster_resources().get("GPU", 0)
print(f"Total GPUs  : {total_gpus:.0f}")
assert total_gpus >= NUM_GPU_WORKERS, "GPUs missing — check spark.task.resource.gpu.amount = '0'"

## Shared preprocessing + reader

`decode_resize` fuses JPEG-decode → resize → normalize into the read path and runs on CPU actors, in parallel with GPU training. `read_format` builds the branch-specific dataset with **column projection** — only `image` + `category` are read; `caption`, `embedding`, and the numeric metadata are skipped. Projected reads are where Lance's blob isolation shows up: metadata columns are never touched, and image bytes are byte-offset addressed.

In [ ]:
import numpy as np

CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


def decode_resize(batch, img_size, cat_to_idx):
    """Fused decode → resize → normalize. Returns fixed-size CHW float32 + int label."""
    import io
    from PIL import Image

    imgs, labels = [], []
    for jpeg, cat in zip(batch["image"], batch["category"]):
        img = Image.open(io.BytesIO(bytes(jpeg))).convert("RGB").resize((img_size, img_size))
        arr = (np.asarray(img, dtype=np.float32) / 255.0).transpose(2, 0, 1)  # CHW
        imgs.append(arr)
        labels.append(cat_to_idx[cat if isinstance(cat, str) else cat.decode()])
    return {"image": np.asarray(imgs, dtype=np.float32),
            "label": np.asarray(labels, dtype=np.int64)}


def read_format(fmt, parquet_path, lance_path, columns):
    """Branch-specific reader with column projection."""
    import ray
    if fmt == "lance":
        return ray.data.read_lance(lance_path, columns=columns)
    return ray.data.read_parquet(parquet_path, columns=columns)

## Section 1 — Data loading throughput

Full pass over each format, decode/resize fused in, no GPU. Run twice per format: **full-column** read (all columns) vs **projected** read (`image` + `category` only). The projected/full gap is the direct measure of column-selective read efficiency.

In [ ]:
import time


def loading_throughput(fmt, parquet_path, lance_path, columns, img_size, cat_to_idx, batch_size):
    import time, ray
    ds = read_format(fmt, parquet_path, lance_path, columns).map_batches(
        decode_resize, fn_kwargs={"img_size": img_size, "cat_to_idx": cat_to_idx},
        batch_size=batch_size,
    )
    t0 = time.time()
    n = 0
    for b in ds.iter_batches(batch_size=batch_size, batch_format="numpy"):
        n += len(b["label"])
    dt = time.time() - t0
    return {"samples_per_sec": round(n / dt, 1), "total": n, "elapsed_s": round(dt, 2)}


loading_results = {}
PROJECTED = ["image", "category"]
FULL      = ["id", "image", "caption", "embedding", "category", "brightness", "quality"]

for fmt in ["parquet", "lance"]:
    loading_results[(fmt, "projected")] = loading_throughput(
        fmt, parquet_path, lance_path, PROJECTED, IMG_SIZE, CAT_TO_IDX, BATCH_SIZE)
    loading_results[(fmt, "full")] = loading_throughput(
        fmt, parquet_path, lance_path, FULL, IMG_SIZE, CAT_TO_IDX, BATCH_SIZE)
    for read in ["projected", "full"]:
        r = loading_results[(fmt, read)]
        print(f"  {fmt:8s} {read:10s}: {r['samples_per_sec']:>10,.1f} samples/s "
              f"({r['total']:,} in {r['elapsed_s']}s)")

## Section 2 — Training throughput

A ResNet-50 classifier (ImageNet-pretrained backbone, fresh head sized to the synthetic categories) trained with `TorchTrainer` + DDP across the A10 workers. Shuffled read per epoch (`random_shuffle`) — the access pattern that separates the formats; a sequential scan looks similar for both.

Two runs per format:
- **dummy** — forward/backward skipped, so throughput = pure data-loading ceiling.
- **real** — full ResNet-50 step, so the bottleneck is attributed between I/O and compute.

In [ ]:
def train_fn_per_worker(config):
    """Per-worker DDP loop. Captures samples/sec, time-to-first-batch, and per-batch
    latency percentiles (p50/p95/p99), then reports rank-0 metrics to MLflow."""
    import time
    import numpy as np
    import torch
    import torch.nn as nn
    import ray.train, ray.train.torch
    from torchvision.models import resnet50, ResNet50_Weights
    from mlflow.tracking import MlflowClient

    device   = ray.train.torch.get_device()
    rank     = ray.train.get_context().get_world_rank()
    dummy    = config["dummy"]
    n_epochs = config["num_epochs"]
    bs       = config["batch_size"]

    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)   # transfer learning
    model.fc = nn.Linear(model.fc.in_features, config["num_classes"])
    model = ray.train.torch.prepare_model(model.to(device))
    opt   = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    lossf = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()

    shard  = ray.train.get_dataset_shard("train")
    client = MlflowClient() if rank == 0 else None

    for epoch in range(n_epochs):
        model.train()
        t_epoch = time.time()
        n, ttfb, batch_ms = 0, None, []
        for batch in shard.iter_torch_batches(batch_size=bs, dtypes=torch.float32, device=device):
            t_b = time.time()
            imgs = batch["image"]
            labels = batch["label"].long()
            if not dummy:
                with torch.cuda.amp.autocast():
                    loss = lossf(model(imgs), labels)
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                _ = imgs.mean()   # touch the data, skip compute
            if ttfb is None:
                ttfb = time.time() - t_epoch
            batch_ms.append((time.time() - t_b) * 1000)
            n += imgs.shape[0]

        dt = time.time() - t_epoch
        p = np.percentile(batch_ms, [50, 95, 99])
        metrics = {
            "samples_per_sec": n / dt, "epoch_wall_s": dt, "time_to_first_batch_s": ttfb,
            "batch_ms_p50": float(p[0]), "batch_ms_p95": float(p[1]), "batch_ms_p99": float(p[2]),
        }
        if client is not None:
            for k, v in metrics.items():
                client.log_metric(config["mlflow_run_id"], k, v, step=epoch)
        ray.train.report(metrics)

In [ ]:
import mlflow
import pandas as pd
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig

mlflow.set_experiment(mlflow_exp)
training_results = {}

for fmt in ["parquet", "lance"]:
    for dummy in [True, False]:
        mode = "dummy" if dummy else "real"

        ds = read_format(fmt, parquet_path, lance_path, PROJECTED).map_batches(
            decode_resize, fn_kwargs={"img_size": IMG_SIZE, "cat_to_idx": CAT_TO_IDX},
            batch_size=BATCH_SIZE,
        ).random_shuffle()   # shuffled random-access read per epoch

        with mlflow.start_run(run_name=f"{fmt}_{mode}_{size}") as run:
            mlflow.log_params({"format": fmt, "mode": mode, "dataset_size": size,
                               "model": "resnet50", "batch_size": BATCH_SIZE,
                               "num_epochs": NUM_EPOCHS, "num_gpu_workers": NUM_GPU_WORKERS})
            trainer = TorchTrainer(
                train_loop_per_worker=train_fn_per_worker,
                train_loop_config={"dummy": dummy, "num_epochs": NUM_EPOCHS,
                                   "batch_size": BATCH_SIZE, "num_classes": len(CATEGORIES),
                                   "mlflow_run_id": run.info.run_id},
                scaling_config=ScalingConfig(num_workers=NUM_GPU_WORKERS, use_gpu=True),
                datasets={"train": ds},
                run_config=RunConfig(storage_path=ray_tmp_path),
            )
            result = trainer.fit()
            training_results[(fmt, mode)] = result.metrics
            mlflow.log_metrics({k: v for k, v in result.metrics.items()
                                if isinstance(v, (int, float))})
        print(f"  {fmt:8s} {mode:6s}: {result.metrics.get('samples_per_sec', 0):>10,.1f} samples/s")

## Section 3 — Compare

Side-by-side across loading and training. Expected divergence (per the README): **projected-column reads** and **shuffled random-access training throughput** — not raw sequential scan. If the *dummy* gap is large but the *real* gap shrinks, the format difference is being masked by GPU compute — the tell to scale down the model or up the data tier.

In [ ]:
rows = []
for fmt in ["parquet", "lance"]:
    rows.append({
        "format": fmt,
        "load_projected_sps": loading_results[(fmt, "projected")]["samples_per_sec"],
        "load_full_sps":      loading_results[(fmt, "full")]["samples_per_sec"],
        "train_dummy_sps":    round(training_results[(fmt, "dummy")].get("samples_per_sec", 0), 1),
        "train_real_sps":     round(training_results[(fmt, "real")].get("samples_per_sec", 0), 1),
        "real_ttfb_s":        round(training_results[(fmt, "real")].get("time_to_first_batch_s", 0), 2),
        "real_p99_ms":        round(training_results[(fmt, "real")].get("batch_ms_p99", 0), 1),
    })
summary = pd.DataFrame(rows)
display(summary)

l = summary[summary.format == "lance"].iloc[0]
p = summary[summary.format == "parquet"].iloc[0]
if p.train_real_sps > 0:
    print(f"\nLance vs Parquet @ {size}:")
    print(f"  projected read : {l.load_projected_sps / max(1, p.load_projected_sps):.2f}x")
    print(f"  train (real)   : {l.train_real_sps / p.train_real_sps:.2f}x")

## Deferred training metrics

Left out of this draft — they need node/GPU-level instrumentation rather than in-loop timing:

- **GPU utilization %** — via `nvidia-smi` / DCGM sampled during the run (the direct data-starvation signal; samples/sec is the in-loop proxy used here).
- **Object-store spill events / disk IOPS** — from the Ray dashboard during shuffled reads.
- **Actor-pool utilization** (saturated vs waiting on IO) and **CPU utilization on preprocess actors** — Ray dashboard timelines.